[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/elliptic_physics/s2_physics.ipynb)

# Rigid Bodies on the Sphere

This example simulates rigid ellipses spinning and colliding on the 2-sphere. Because rigid motions of the sphere are rotations of ambient three-dimensional space, every body turns about the sphere's centre, and two bodies that meet push each other apart along their shared normal great circle while conserving kinetic energy and total world-frame momentum.

Each body's shape is modelled as a dual quadric mapping a great-circle plane to its pole, while its inverse primal form maps a sphere point to its polar great circle and evaluates negative inside. Sandwiching places any primal form in the world frame as `motors >> forms(motors << Point)`, mass points inside the body assemble its inertia as a ***duality map*** `((masses * points).regressive(points.commutator(Rate))).sum(axis=-1)` (the term ***duality map*** here means a linear map from an extrinsic space to its dual intrinsic space, or the reverse), and maximizing the concave least eigenvalue across positive blends of two primal forms yields their overlap margin and deepest point.

In matrix Newton–Euler and conic notation on $S^2$, placing a $3 \times 3$ conic matrix $C$ by a rotation matrix $R \in \mathrm{SO}(3)$ requires the congruence $R C R^{\mathsf T}$, mass-cloud inertia builds the body-frame tensor $I = \sum_k m_k (|\mathbf r_k|^2 \mathbf 1 - \mathbf r_k \mathbf r_k^{\mathsf T})$, and body-frame angular velocity evolves under Euler's cross-product equation $I \dot{\boldsymbol\omega} = (I \boldsymbol\omega) \times \boldsymbol\omega + \boldsymbol\tau$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import Spherical3D
from examples.util.animation import save_figure, show_states
from examples.quadrics.elliptic_physics import render, scenarios

np.set_printoptions(precision=4, suppress=True)

ga = Spherical3D
ctx = NumpyContext(ga, dtype=np.float64)
mv = ctx.multivector

Scalar = ga.gatype.scalar()                                                     # Scalar+
Point = ga.gatype.antivector()                                                  # Antivector-
Plane = ga.gatype.vector()                                                      # Vector+
Rate = ga.gatype.bivector()                                                     # Bivector+
Momentum = ga.gatype.antibivector()                                             # Antibivector-
Motor = ga.gatype.rotor()                                                       # Rotor+
# A primal quadric maps a point to its polar plane and evaluates negative inside; a dual quadric maps a plane to its pole.
Quadric = ga.gatype((Plane, Point))                                             # Plane+ <- Point-
DualQuadric = ga.gatype((Point, Plane))                                         # Point- <- Plane+
Inertia = ga.gatype((Momentum, Rate))                                           # Momentum- <- Rate+
InverseInertia = ga.gatype((Rate, Momentum))                                    # Rate+ <- Momentum-

# Golden-section iterations bracketing the optimal quadric blend angle to within 0.005 rad.
n_golden_steps = 12

# Unit poles of the coordinate planes, with the final pole serving as each body's reference centre.
basis = mv.vector(np.eye(ga.dimension)).dual().normalized()                     # [3] Point-
origin = basis[-1]                                                              # [] Point-

## Spherical ellipses as dual and primal quadrics

Weighting the three coordinate-axis dyads by the squared tangents of the angular half-widths and minus one at the reference pole constructs a central ellipse as a dual quadric that sends each great-circle plane to its pole. Inverting the dual quadric gives the primal form, which pairs negatively with points inside the ellipse so every front-hemisphere sample tests membership directly.

In projective conic matrix notation, the dual and primal conic matrices are inverses up to scale, $C^* \propto C^{-1}$, and point membership tests the quadratic sign $\mathbf x^{\mathsf T} C \mathbf x < 0$.

In [ ]:
ellipse_half_widths = scenarios.ellipse_half_widths
ellipse_turn = scenarios.ellipse_turn

In [ ]:
def diagonal_quadric(diagonal: np.ndarray) -> DualQuadric:
    """Coordinate poles weighted by `diagonal` (`[..., 3]`) sum into diagonal dual quadrics (`[...] Point- <- Plane+`)."""
    return (basis * (basis & Plane) * diagonal).sum(axis=-1)                    # [...] Point- <- Plane+


def ellipsoid_quadric(half_widths: np.ndarray) -> DualQuadric:
    """Tangent `half_widths` (`[..., 2]`) around `origin` form central ellipsoid dual quadrics (`[...] Point- <- Plane+`)."""
    weights = np.concatenate([half_widths**2, -np.ones_like(half_widths[..., :1])], axis=-1)
    return diagonal_quadric(weights)                                            # [...] Point- <- Plane+

In [ ]:
def world_quadrics(motors: Motor, forms: Quadric) -> Quadric:
    """Sandwiching body-frame `forms` (`[...] Plane+ <- Point-`) by `motors` (`[...] Motor+`) places the primal quadrics in the world frame."""
    return motors >> forms(motors << Point)                                     # [...] Plane+ <- Point-


ellipse_dual = ellipsoid_quadric(np.tan(ellipse_half_widths))                   # [] Point- <- Plane+
ellipse_motor = (mv.xy * ellipse_turn).exp()                                    # [] Motor+
ellipse_form = world_quadrics(ellipse_motor, ellipse_dual.inverse())            # [] Plane+ <- Point-

hemisphere_samples = render.hemisphere_points()                                 # [...] Point-
inside_mask = (ellipse_form(hemisphere_samples) & hemisphere_samples) < 0

print(ellipse_form)

ellipse_fig = render.inside_figure(inside_mask)
save_figure(ellipse_fig, "spherical_quadric_ellipse")
ellipse_fig

## Overlap detection and elastic contact impulses

Mass points inside each ellipse determine its body-frame inertia from the regressive product of each point with its commutator velocity under `Rate`, while placing a primal form into a second body's frame reduces contact detection to blending two quadrics. Two interiors where both quadratic forms are negative are disjoint if and only if some positive combination is positive semidefinite, so golden-section maximization of the concave least eigenvalue over blend angle returns a negative margin on overlap and the deepest shared point as its eigenvector; contracting the polar plane there with its dual pole yields the normal contact forque, and an elastic impulse along that forque reverses the closing rate.

In convex-optimization matrix notation, the S-lemma certifies disjointness of $\{\mathbf x : \mathbf x^{\mathsf T} A \mathbf x < 0\}$ and $\{\mathbf x : \mathbf x^{\mathsf T} B \mathbf x < 0\}$ through $\max_{\phi \in (0, \pi/2)} \lambda_{\min}(A + \tan\phi\, B) \ge 0$.

In [ ]:
pair_radians = np.radians(scenarios.pair_half_angles)
pair_points, pair_sample_masses = scenarios.ellipse_samples(
    pair_radians, scenarios.pair_masses, scenarios.n_pair_rings, 10,
)
pair_motors = scenarios.polar_placement(scenarios.pair_polars, scenarios.pair_azimuths)  # [2] Motor+
pair_rates = (
    mv.yz * scenarios.pair_rates[:, 0]
    + mv.zx * scenarios.pair_rates[:, 1]
    + mv.xy * scenarios.pair_rates[:, 2]
)                                                                               # [2] Rate+
pair_colors = render.rgb_colors(scenarios.pair_colours)
first_pair, second_pair = np.array([0]), np.array([1])

In [ ]:
def cloud_inertia(points: Point, masses: Scalar) -> Inertia:
    """Mass-weighted regressive products of `points` (`[..., n_samples] Point-`) with their commutator velocities form body inertias (`[...] Momentum- <- Rate+`)."""
    return ((masses * points).regressive(points.commutator(Rate))).sum(axis=-1)  # [...] Momentum- <- Rate+


def quadric_overlap(first: Quadric, second: Quadric) -> tuple[Scalar, Point]:
    """Golden-section search over positive blends of `first` and `second` (`[...] Plane+ <- Point-`) finds the overlap margin (`[...] Scalar+`) and deepest point (`[...] Point-`)."""
    def least_eigenvalue(phi: np.ndarray) -> Scalar:
        blend = Point & (first + second * np.tan(phi))(Point)                   # [...] Scalar+ <- (Point-, Point-)
        return blend.eigvalsh()[..., 0]                                         # [...] Scalar+

    golden = (np.sqrt(5.0) - 1.0) / 2.0
    pair_shape = np.broadcast_shapes(first.shape, second.shape)
    lower, upper = np.broadcast_to(0.0, pair_shape), np.broadcast_to(np.pi / 2, pair_shape)
    left_probe = upper - golden * (upper - lower)
    right_probe = lower + golden * (upper - lower)
    left_value, right_value = least_eigenvalue(left_probe), least_eigenvalue(right_probe)  # [...] Scalar+, [...] Scalar+
    for _ in range(n_golden_steps):
        keep_left = left_value > right_value
        lower = np.where(keep_left, lower, left_probe)
        upper = np.where(keep_left, right_probe, upper)
        left_probe = upper - golden * (upper - lower)
        right_probe = lower + golden * (upper - lower)
        fresh = least_eigenvalue(np.where(keep_left, left_probe, right_probe))  # [...] Scalar+
        left_value, right_value = (
            right_value.at[keep_left].set(fresh[keep_left]),
            fresh.at[keep_left].set(left_value[keep_left]),
        )
    eigenvalues, eigenvectors = (Point & (first + second * np.tan((lower + upper) / 2))(Point)).eigh()
    return eigenvalues[..., 0], eigenvectors[..., 0]

In [ ]:
def contact_forques(
    first_forms: Quadric,
    first_duals: DualQuadric,
    second_forms: Quadric,
    first_motors: Motor,
    second_motors: Motor,
) -> tuple[Scalar, Momentum, Momentum]:
    """Overlap detection between `first_forms` and `second_forms` returns the margin (`[...] Scalar+`) and the contact forques (`[...] Momentum-`) in both body frames."""
    relative = first_motors.inverse() * second_motors                           # [...] Motor+
    margin, deepest = quadric_overlap(first_forms, relative >> second_forms(relative << Point))  # [...] Scalar+, [...] Point-
    # Contracting the first body's unit polar plane at the deepest point with its pole forms the normal contact line.
    contact_plane = first_forms(deepest).normalized()                           # [...] Plane+
    contact_point = first_duals(contact_plane)                                  # [...] Point-
    first_forque = contact_plane | contact_point                                # [...] Momentum-
    second_forque = relative << first_forque                                    # [...] Momentum-
    return margin, first_forque, second_forque


pair_duals = ellipsoid_quadric(np.tan(pair_radians))                            # [2] Point- <- Plane+
pair_forms = pair_duals.inverse()                                               # [2] Plane+ <- Point-
pair_inertias = cloud_inertia(pair_points, pair_sample_masses)                  # [2] Momentum- <- Rate+
pair_inverse_inertias = pair_inertias.inverse()                                 # [2] Rate+ <- Momentum-
pair_momenta = pair_inertias(pair_rates)                                        # [2] Momentum-

pair_margin, one_forque, other_forque = contact_forques(
    pair_forms[first_pair],
    pair_duals[first_pair],
    pair_forms[second_pair],
    pair_motors[first_pair],
    pair_motors[second_pair],
)                                                                               # [1] Scalar+, [1] Momentum-, [1] Momentum-
one_response = pair_inverse_inertias[first_pair](one_forque)                    # [1] Rate+
other_response = pair_inverse_inertias[second_pair](other_forque)               # [1] Rate+
closing_before = (one_response & pair_momenta[first_pair]) - (other_response & pair_momenta[second_pair])  # [1] Scalar+
pair_compliance = (one_response & one_forque) + (other_response & other_forque)  # [1] Scalar+
pair_impulse = -2 * closing_before / pair_compliance                            # [1] Scalar+
first_after = pair_momenta[first_pair] + one_forque * pair_impulse              # [1] Momentum-
second_after = pair_momenta[second_pair] - other_forque * pair_impulse          # [1] Momentum-
closing_after = (one_response & first_after) - (other_response & second_after)  # [1] Scalar+

print("margin:", pair_margin.to_array())
print("closing rate before:", closing_before.to_array())

contact_fig = render.contact_figure(world_quadrics(pair_motors, pair_forms), pair_colors)
save_figure(contact_fig, "spherical_quadric_contact")
contact_fig

## Colliding crowd and intermediate-axis tumbling

Advancing placements and body-frame momenta with a Lie midpoint step preserves free-body momentum while bounding-radius geodesic filtering restricts overlap tests to nearby pairs at each substep. Streaming seven ellipses from needles to discs shows repeated elastic rebounds across the front hemisphere, while a single elongated oval started near its intermediate axis flips periodically back and forth without external torque. Generating both animations takes a few seconds.

In [ ]:
crowd_colors, crowd_motors, crowd_momenta, crowd_duals, crowd_forms, crowd_inverse_inertias, crowd_radii = scenarios.crowded_scene()
tumbling_colors, tumbling_motors, tumbling_momenta, tumbling_duals, tumbling_forms, tumbling_inverse_inertias, tumbling_radii = scenarios.tumbling_scene()
s2_dt = scenarios.s2_dt
n_s2_frames = scenarios.n_s2_frames
n_s2_substeps = scenarios.n_s2_substeps
n_tumbling_frames = scenarios.n_tumbling_frames
n_tumbling_substeps = scenarios.n_tumbling_substeps
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def kinetic_energy(rates: Rate, momenta: Momentum) -> Scalar:
    """Half the regressive pairing of body-frame `rates` (`[n_bodies] Rate+`) with `momenta` (`[n_bodies] Momentum-`) sums to total kinetic energy (`[] Scalar+`)."""
    return (rates & momenta).sum() / 2                                          # [] Scalar+


def world_momentum(motors: Motor, momenta: Momentum) -> Momentum:
    """Carrying body-frame `momenta` (`[n_bodies] Momentum-`) by `motors` (`[n_bodies] Motor+`) sums to total world-frame momentum (`[] Momentum-`)."""
    return (motors >> momenta).sum(axis=0)                                      # [] Momentum-

In [ ]:
def midpoint_step(
    motors: Motor,
    momenta: Momentum,
    inverse_inertias: InverseInertia,
    dt: float,
) -> tuple[Motor, Momentum]:
    """Advancing `motors` and body-frame `momenta` by `dt` with a Lie midpoint step updates placements and body momenta (`[...] Motor+, [...] Momentum-`)."""
    # Step halfway along the present body rate, evaluate the midpoint rate, and advance across the full step.
    half_step = (inverse_inertias(momenta) * (dt / 4)).exp()                    # [...] Motor+
    midpoint = (motors * half_step).normalized()                                # [...] Motor+
    midpoint_rate = inverse_inertias((motors.inverse() * midpoint) << momenta)  # [...] Rate+
    whole_step = (midpoint_rate * (dt / 2)).exp()                               # [...] Motor+
    next_motors = (motors * whole_step).normalized()                            # [...] Motor+
    return next_motors, (motors.inverse() * next_motors) << momenta


def candidate_pairs(motors: Motor, bounding_radii: Scalar) -> tuple[np.ndarray, np.ndarray]:
    """Geodesic distances between body centres (`[n_bodies] Motor+`) filter upper-triangular index pairs within the sum of `bounding_radii` (`[n_bodies] Scalar+`)."""
    poles = motors >> origin                                                    # [n_bodies] Point-
    separations = (poles[:, None] | poles).abs().clip(0.0, 1.0).arccos()        # [n_bodies, n_bodies] Scalar+
    within_reach = separations < bounding_radii[:, None] + bounding_radii[None, :]
    return np.nonzero(np.triu(within_reach, 1))

In [ ]:
def elastic_collision(
    motors: Motor,
    momenta: Momentum,
    forms: Quadric,
    duals: DualQuadric,
    inverse_inertias: InverseInertia,
    first_indices: np.ndarray,
    second_indices: np.ndarray,
    dt: float,
) -> tuple[Momentum, int]:
    """Sequential elastic impulses across approaching overlapping pairs `(first_indices, second_indices)` update `momenta` (`[n_bodies] Momentum-`) and count applied impulses."""
    margin, first_forque, second_forque = contact_forques(
        forms[first_indices],
        duals[first_indices],
        forms[second_indices],
        motors[first_indices],
        motors[second_indices],
    )
    # Compare against a virtual half-step forward so impulses act only while an overlap is deepening, independent of forque sign.
    half_step = (inverse_inertias(momenta) * (dt / 2)).exp()                    # [n_bodies] Motor+
    ahead = (motors * half_step).normalized()                                   # [n_bodies] Motor+
    relative_ahead = ahead[first_indices].inverse() * ahead[second_indices]     # [...] Motor+
    margin_ahead, _ = quadric_overlap(
        forms[first_indices],
        relative_ahead >> forms[second_indices](relative_ahead << Point),
    )
    approaching = (margin < 0) & (margin_ahead < margin)
    for first_body, second_body, one, other in zip(
        first_indices[approaching],
        second_indices[approaching],
        first_forque[approaching],
        second_forque[approaching],
    ):
        first_response = inverse_inertias[first_body](one)                      # [] Rate+
        second_response = inverse_inertias[second_body](other)                  # [] Rate+
        closing = (first_response & momenta[first_body]) - (second_response & momenta[second_body])  # [] Scalar+
        compliance = (first_response & one) + (second_response & other)         # [] Scalar+
        impulse = -2 * closing / compliance                                     # [] Scalar+
        momenta = (
            momenta.at[first_body]
            .set(momenta[first_body] + one * impulse)
            .at[second_body]
            .set(momenta[second_body] - other * impulse)
        )
    return momenta, int(approaching.sum())


def collision_frames(
    motors: Motor,
    momenta: Momentum,
    forms: Quadric,
    duals: DualQuadric,
    inverse_inertias: InverseInertia,
    bounding_radii: Scalar,
    n_frames: int,
    dt: float,
    n_substeps: int,
) -> Iterator[tuple[Motor, Momentum, int]]:
    """Yields `n_frames` states `(motors, momenta, cumulative_impulses)` separated by `n_substeps` midpoint and collision substeps across `dt`."""
    substep_dt = dt / n_substeps
    impulses = 0
    for _ in range(n_frames):
        yield motors, momenta, impulses
        for _ in range(n_substeps):
            motors, momenta = midpoint_step(motors, momenta, inverse_inertias, substep_dt)
            first_indices, second_indices = candidate_pairs(motors, bounding_radii)
            momenta, applied = elastic_collision(
                motors, momenta, forms, duals, inverse_inertias, first_indices, second_indices, substep_dt,
            )
            impulses += applied


crowd_states = show_states(
    collision_frames(
        crowd_motors,
        crowd_momenta,
        crowd_forms,
        crowd_duals,
        crowd_inverse_inertias,
        crowd_radii,
        n_s2_frames,
        s2_dt,
        n_s2_substeps,
    ),
    render.s2_animation,
    crowd_forms,
    crowd_colors,
    render.painted_resolution,
    name="spherical_quadric_physics",
    duration_ms=frame_duration_ms,
    n_frames=n_s2_frames,
);

In [ ]:
tumbling_states = show_states(
    collision_frames(
        tumbling_motors,
        tumbling_momenta,
        tumbling_forms,
        tumbling_duals,
        tumbling_inverse_inertias,
        tumbling_radii,
        n_tumbling_frames,
        s2_dt,
        n_tumbling_substeps,
    ),
    render.s2_animation,
    tumbling_forms,
    tumbling_colors,
    render.painted_resolution,
    name="spherical_tumbling_oval",
    duration_ms=frame_duration_ms,
    n_frames=n_tumbling_frames,
)
tumbling_rates = stack([tumbling_inverse_inertias(momenta)[0] for _, momenta, _ in tumbling_states])  # [n_tumbling_frames] Rate+
tumbling_fig = render.tumbling_figure(tumbling_rates, s2_dt)
save_figure(tumbling_fig, "spherical_tumbling_oval")
tumbling_fig

Blue traces the body-frame rate component in the intermediate `yz` plane as it reverses sign on each flip, while crimson (`zx`) and amber (`xy`) carry the coupled oscillations about the extreme principal axes.

In [ ]:
# checks
assert bool(inside_mask.any()) and bool((~inside_mask).any())
assert (pair_margin < 0).all()
assert ((closing_after + closing_before).abs() < 1e-12).all()

crowd_energies = stack([kinetic_energy(crowd_inverse_inertias(momenta), momenta) for _, momenta, _ in crowd_states])  # [n_s2_frames] Scalar+
crowd_momenta_norms = stack([world_momentum(motors, momenta).antinorm() for motors, momenta, _ in crowd_states])  # [n_s2_frames] Scalar+
assert crowd_states[-1][2] > 0
assert (((crowd_energies.max() - crowd_energies.min()) / crowd_energies[0].abs()) < 1e-2).all()
assert (((crowd_momenta_norms.max() - crowd_momenta_norms.min()) / crowd_momenta_norms[0]) < 1e-11).all()

intermediate_spin = tumbling_rates & mv.yz.dual()                               # [n_tumbling_frames] Scalar+
positive_spin = intermediate_spin > 0
assert int(np.count_nonzero(positive_spin[1:] != positive_spin[:-1])) >= 2